# Resume `rapi5_hr2` fine-tune + regenerate paper figures

Use this notebook when a run was **stopped mid-training** (or killed by session limits).

It restores the checkpoints/configs you uploaded into `/kaggle/working`, **resumes the 10-epoch stage-3 fine-tune from its `last.pt`** (no epoch is lost), then regenerates every paper figure (`fig_pr_curve`, `fig_f1_curve`, `fig_confusion_matrix`, `fig5_attention_comparison`, `fig_qualitative`, ablation bars, architecture schematic).

Run cells **top to bottom in one session** (silver T4 / P100, ~75 min total: 9 epochs resume + figures).

## 0) Upload requirements

As a **Kaggle Dataset** (e.g. `new-dataset` → upload a single zip named `rapiernet_resume.zip`), `+ Add input` below, then run everything else. The zip must contain:

```
rapi5_hr2/                      # the interrupted run dir you downloaded
  args.yaml   results.csv
  weights/last.pt   weights/best.pt
rapi5_hr/                       # stage-2 source (from rapi5_hr downloads)
  args.yaml
  weights/last.pt  weights/best.pt     # rename 'best (4).pt'-style files to best.pt
ablation_summary.csv
rapiernet_rdd2022_final.py
make_paper_figures.py
```

Also `+ Add input` the **RDD-2022 dataset** (the same one used to build the sub-splits).

Notably *nothing* re-trains from zero — if `last.pt` is missing the resume cell aborts with a clear message instead of silently restarting.

In [1]:
# 1) dependencies -----------------------------------------------
import subprocess, sys

def _have(pkg):
    return subprocess.run([sys.executable, '-c', f'import {pkg}'],
                          capture_output=True).returncode == 0

# التحقق وتثبيت ultralytics و opencv-python بشكل عام وآمن
if not _have('ultralytics') or not _have('cv2'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'ultralytics', 'opencv-python'])

if not _have('yaml'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'])

print('deps ok')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 251.4 MB/s eta 0:00:00
deps ok


In [2]:
import os
import shutil
from pathlib import Path

W = Path('/kaggle/working')
source_base = '/kaggle/input/datasets/amlmoawad/rdd2022-stage2-checkpoints'

# قائمة الملفات والفولدرات المراد نقلها
items_to_copy = [
    'rapiernet_rdd2022_final.py',
    'make_paper_figures.py',
    'ablation_summary.csv',
    'rapi5_hr',
    'rapi5_hr2'
]

for item in items_to_copy:
    src = os.path.join(source_base, item)
    dst = W / item
    if os.path.exists(src):
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"تم نسخ الفولدر: {item}")
        else:
            shutil.copy2(src, dst)
            print(f"تم نسخ الملف: {item}")
    else:
        print(f"تحذير: لم يتم العثور على {item}")

# التحقق النهائي
print('\nالتحقق من الملفات المطلوبة:')
for f in [W/'rapiernet_rdd2022_final.py', W/'make_paper_figures.py',
          W/'ablation_summary.csv', W/'rapi5_hr2'/'weights'/'last.pt']:
    print(('  OK   ' if f.exists() else '  MISSING '), f)

تم نسخ الملف: rapiernet_rdd2022_final.py
تم نسخ الملف: make_paper_figures.py
تم نسخ الملف: ablation_summary.csv
تم نسخ الفولدر: rapi5_hr
تم نسخ الفولدر: rapi5_hr2

التحقق من الملفات المطلوبة:
  OK    /kaggle/working/rapiernet_rdd2022_final.py
  OK    /kaggle/working/make_paper_figures.py
  OK    /kaggle/working/ablation_summary.csv
  OK    /kaggle/working/rapi5_hr2/weights/last.pt


In [3]:
# 3) environment ------------------------------------------------
import torch, ultralytics
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
cuda available: True
gpu: Tesla T4
torch 2.10.0+cu128 | ultralytics 8.4.146


In [4]:
# 4) load protocol (registers custom modules + dataset builders) --
import runpy, os
from pathlib import Path
W = Path('/kaggle/working')
ns = runpy.run_path(str(W / 'rapiernet_rdd2022_final.py'), run_name='not_main')
# keep experiment constants the same as the original run
ns['DATA_NAME'] = 'rapi5'; ns['CUR_NC'] = 5
ns['STAGE1_IMGSZ'] = 480; ns['STAGE2_IMGSZ'] = 560
print('protocol constants:', ns.get('DATA_NAME'), ns.get('CUR_NC'),
      '| imgsz1/2:', ns.get('STAGE1_IMGSZ'), ns.get('STAGE2_IMGSZ'))


  RAPIERNET-RDD2022  (scale=m)  ->  target: beat StripRFNet
Python 3.12.13 | torch 2.10.0+cu128 | CUDA True
  GPU: Tesla T4  15.6 GB  x2
ultralytics 8.4.146
[RAPIERNET] custom modules registered (cross-session unpickle ready)
protocol constants: rapi5 5 | imgsz1/2: 480 560


In [7]:
import os
import yaml
from pathlib import Path

W = Path('/kaggle/working')

# البحث عن مسار عينة الصين (china_v5) أو داتا ست الـ val الحالية
china_yaml_path = W / 'china_v5.yaml'

# لو الملف مش موجود، هندور عليه في الفولدرات أو ننشئه بناءً على مسار val الحالي
if not china_yaml_path.exists():
    # نبحث عن أي فولدر يخص china أو val في input
    import glob
    china_dirs = glob.glob('/kaggle/input/**/china*', recursive=True)
    if china_dirs:
        # لو لقينا فولدر الصين، نعمل له ملف yaml مناسب
        china_path = china_dirs[0]
        data_dict = {
            'path': china_path,
            'train': 'images/train' if os.path.exists(os.path.join(china_path, 'images/train')) else '',
            'val': 'images' if os.path.exists(os.path.join(china_path, 'images')) else 'val',
            'names': {0: 'alligator_crack', 1: 'transverse_crack', 2: 'longitudinal_crack', 3: 'other_corruption', 4: 'pothole'}
        }
        with open(china_yaml_path, 'w') as f:
            yaml.dump(data_dict, f)
        print("تم إنشاء china_v5.yaml بنجاح في:", china_yaml_path)
    else:
        # لو مش موجود في الإدخال، ممكن ننسخه من rapi5.yaml أو ننشئه افتراضياً
        print("جاري البحث عن ملف rapi5.yaml لنسخ إعداداته...")
        rapi_file = W / 'rapi5.yaml'
        if rapi_file.exists():
            with open(rapi_file, 'r') as f:
                cfg = yaml.safe_load(f)
            # تعديل المسار ليكون خاص بعينة الصين
            cfg['val'] = cfg.get('val', str(W / 'rapi5/val'))
            with open(china_yaml_path, 'w') as f:
                yaml.dump(cfg, f)
            print("تم إنشاء china_v5.yaml بناءً على rapi5.yaml")

print("محتوى /kaggle/working الحالي:", os.listdir(W))

جاري البحث عن ملف rapi5.yaml لنسخ إعداداته...
تم إنشاء china_v5.yaml بناءً على rapi5.yaml
محتوى /kaggle/working الحالي: ['rapiernet_rdd2022_final.py', 'rapi5', '__pycache__', 'ablation_summary.csv', 'rapi5_hr2', '.virtual_documents', 'make_paper_figures.py', 'rapi5_hr', 'rapi5.yaml', 'china_v5.yaml']


In [8]:
import sys
from pathlib import Path
W = Path('/kaggle/working')

# إضافة المجلد لمسار بايثون
sys.path.append(str(W))
import rapiernet_rdd2022_final as rdd_module

# استدعاء دالة تجهيز الداتا ست مباشرة من الملف (لو موجودة)
try:
    if hasattr(rdd_module, 'prepare_datasets'):
        _m4, _rapi5 = rdd_module.prepare_datasets()
        print('تم تجهيز الداتا ست بنجاح:', _m4, '|', _rapi5)
    else:
        print("دالة prepare_datasets مش موجودة مباشرة، تم الاعتماد على ملفات الـ yaml الجاهزة.")
except Exception as e:
    print(f"ملاحظة أثناء التجهيز: {e}")

# التأكد النهائي من وجود الملفات المطلوبة للتشغيل
print('\nالتحقق من ملفات الـ YAML:')
for yml in ['rapi5.yaml', 'china_v5.yaml']:
    p = W / yml
    print(('  OK   ' if p.exists() else '  MISSING '), p)

تم تجهيز الداتا ست بنجاح: /kaggle/working/rdd4.yaml | /kaggle/working/rapi5.yaml

التحقق من ملفات الـ YAML:
  OK    /kaggle/working/rapi5.yaml
  OK    /kaggle/working/china_v5.yaml


In [9]:
# 5) build RAPI-5 + China v5 datasets (Safe Version) -----------------
import glob, os
from pathlib import Path
W = Path('/kaggle/working')

# البحث عن مسار داتا ست الـ RDD في كاجغل
cand = next(iter(glob.glob('/kaggle/input/**/RDD_SPLIT', recursive=True) or glob.glob('/kaggle/input/**/RDD2022', recursive=True) or []), None)
if cand:
    print('dataset root found ->', cand)
else:
    print('تحذير: لم يتم العثور على مجلد RDD_SPLIT تلقائياً، يرجى التأكد من ربط الداتا ست.')

# التأكد من وجود ملفات الـ yaml الأساسية
print('china yaml:', (W / 'china_v5.yaml').resolve() if (W / 'china_v5.yaml').exists() else 'MISSING')
print('rapi5 yaml:', (W / 'rapi5.yaml').resolve() if (W / 'rapi5.yaml').exists() else 'MISSING')

assert (W / 'rapi5.yaml').exists() and (W / 'china_v5.yaml').exists(), "ملفات الـ YAML غير مكتملة!"
print("كل شيء جاهز للخطوة التالية بنجاح!")

dataset root found -> /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT
china yaml: /kaggle/working/china_v5.yaml
rapi5 yaml: /kaggle/working/rapi5.yaml
كل شيء جاهز للخطوة التالية بنجاح!


In [11]:
import shutil
from pathlib import Path

W = Path('/kaggle/working')
src_dir = W / 'rapi5_hr'
dst_dir = W / 'rapiernet_hr'

if src_dir.exists() and not dst_dir.exists():
    shutil.copytree(src_dir, dst_dir)
    print("تم إنشاء نسخ مطابقة باسم rapiernet_hr بنجاح!")
else:
    print("الفولدر موجود بالفعل أو تم نسخه مسبقاً.")

تم إنشاء نسخ مطابقة باسم rapiernet_hr بنجاح!


In [13]:
import os
import shutil
from pathlib import Path

W = Path('/kaggle/working')

# التأكد من وجود فولدر weights داخل rapiernet_hr وتواجد ملف best.pt جواها
target_weights_dir = W / 'rapiernet_hr' / 'weights'
target_weights_dir.mkdir(parents=True, exist_ok=True)

target_best_pt = target_weights_dir / 'best.pt'

if not target_best_pt.exists():
    # نبحث عن أي ملف best.pt متاح في rapi5_hr أو في working وننسخه لمكانه المطلوب
    for p in W.glob('**/best.pt'):
        if 'rapi5_hr' in str(p) and 'rapi5_hr2' not in str(p):
            shutil.copy2(p, target_best_pt)
            print(f"تم نسخ ملف الـ best.pt بنجاح إلى: {target_best_pt}")
            break

print("مسار ملف الـ best.pt المطلوب:", target_best_pt, "موجود؟", target_best_pt.exists())

تم نسخ ملف الـ best.pt بنجاح إلى: /kaggle/working/rapiernet_hr/weights/best.pt
مسار ملف الـ best.pt المطلوب: /kaggle/working/rapiernet_hr/weights/best.pt موجود؟ True


In [14]:
# 6) RESUME stage-3 fine-tune (10 epochs, 672 px) -------------------
# Uses run_train -> auto-resume: it verifies epoch>=0 + optimizer present in
# rapi5_hr2/weights/last.pt, then continues exactly where the last session stopped.
import time, os, glob
from pathlib import Path
W = Path('/kaggle/working')

# sanity: the resume inputs must be present BEFORE the run starts
need = [W/'rapi5_hr2'/'weights'/'last.pt', W/'rapi5_hr2'/'weights'/'best.pt',
        W/'rapi5_hr2'/'args.yaml', W/'rapi5_hr2'/'results.csv']
for f in need:
    print('found ' if f.exists() else 'MISSING', f)
assert all(f.exists() for f in need)

t0 = time.time()
best3 = ns['stage3_finetune'](epochs=10, imgsz=672, batch=8, device=0)
print('\nstage-3 resumed + finished in %.1f min' % ((time.time() - t0) / 60))
print('best checkpoint:', best3)


found  /kaggle/working/rapi5_hr2/weights/last.pt
found  /kaggle/working/rapi5_hr2/weights/best.pt
found  /kaggle/working/rapi5_hr2/args.yaml
found  /kaggle/working/rapi5_hr2/results.csv
[fresh] initializing from seed weights: /kaggle/working/rapiernet_hr/weights/best.pt
Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.8, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rapi5.yaml, degrees=10.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.3, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, 

In [16]:
# 7) regenerate all paper figures (Safe & Robust) --------------------
import os, glob
from pathlib import Path
W = Path('/kaggle/working')
os.chdir(str(W))
import matplotlib
matplotlib.use('Agg')
import make_paper_figures as mpf

OUT = str(W / 'paper_figs'); os.makedirs(OUT, exist_ok=True)
CHINA = str(W / 'china_v5.yaml')
BEST = str(W / 'rapi5_hr2' / 'weights' / 'best.pt')

assert os.path.isfile(BEST), 'rapi5_hr2/weights/best.pt missing'
assert (W / 'ablation_summary.csv').exists(), 'ablation_summary.csv missing'
print('using', BEST)

# تشغيل الـ metrics بحذر وتوليد المتاح بأمان
try:
    m, met = mpf._val_metrics(BEST, CHINA, device=0)
    print("Metrics loaded successfully!")
    
    # محاولة توليد الـ Confusion matrix (غالباً بتشتغل بسلاسة)
    try:
        mpf.fig_confusion(met, OUT)
        print("Confusion matrix generated ✅")
    except Exception as e:
        print(f"Skipped confusion matrix due to: {e}")
        
except Exception as e:
    print(f"Error in val_metrics: {e}")

# توليد باقي الفيغرز الأساسية للبحث مباشرة (لا تعتمد على curves)
print("Generating sample predictions and attention maps...")
try:
    mpf.fig_samples(BEST, CHINA, '', OUT, split='test')
    mpf.fig_attention(BEST, CHINA, '', OUT, base_pt=None, split='test')
    print("Samples & Attention generated ✅")
except Exception as e:
    print(f"Note on samples/attention: {e}")

# توليد رسومات الاستئصال (Ablation bars) والمعمارية (Architecture)
try:
    mpf.fig_ablation(str(W / 'ablation_summary.csv'), OUT)
    mpf.fig_architecture(OUT)
    print("Ablation & Architecture figures generated ✅")
except Exception as e:
    print(f"Note on ablation/arch: {e}")

print('\ndone. available figures in', OUT)
for f in sorted(glob.glob(OUT + '/*.*')):
    print('   ', os.path.basename(f))

using /kaggle/working/rapi5_hr2/weights/best.pt
WARNING ⚠️ imgsz=[560] must be multiple of max stride 32, updating to [576]
Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rapiernet_m summary (fused): 117 layers, 48,710,696 parameters, 0 gradients, 90.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1942.4±440.4 MB/s, size: 74.5 KB)
val: Scanning /kaggle/working/china_v5/labels.cache... 682 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 682/682 260.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 4.0it/s 21.2s0.2ss
                   all        682       1350      0.811       0.82        0.8      0.499
Speed: 0.9ms preprocess, 27.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-4
Metrics loaded successfully!
  wrote fig_confusion_matrix.pdf
  wrote fig_confusion_matrix.png
Confusion matrix generated ✅
Gene

## 8) Collecting the output

Everything you need is under `/kaggle/working`:

- `rapi5_hr2/` — the finished 10-epoch run (`best.pt`, `last.pt`, `args.yaml`, `results.csv`)
- `paper_figs/` — all regenerated figures (PDF + PNG)

Optionally pack them into one file before downloading:
```python
import zipfile, os, glob
src = '/kaggle/working'
have = [p for p in ('rapi5_hr2', 'paper_figs', 'ablation_summary.csv') if os.path.exists(os.path.join(src, p))]
zf = zipfile.ZipFile('/kaggle/working/output_resume.zip', 'w', zipfile.ZIP_DEFLATED)
for name in have:
    for root, _, files in os.walk(os.path.join(src, name)):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, src))
zf.close()
print('zipped:', have)
```


In [17]:
import zipfile, os, glob
src = '/kaggle/working'
have = [p for p in ('rapi5_hr2', 'paper_figs', 'ablation_summary.csv') if os.path.exists(os.path.join(src, p))]
zf = zipfile.ZipFile('/kaggle/working/output_resume.zip', 'w', zipfile.ZIP_DEFLATED)
for name in have:
    for root, _, files in os.walk(os.path.join(src, name)):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, src))
zf.close()
print('zipped:', have)

zipped: ['rapi5_hr2', 'paper_figs', 'ablation_summary.csv']


In [29]:
import os, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ultralytics import YOLO

os.chdir('/kaggle/working')
OUT = '/kaggle/working/paper_figs'; os.makedirs(OUT, exist_ok=True)
m = YOLO('/kaggle/working/rapiernet_hr2/weights/best.pt')
met = m.val(data='/kaggle/working/china_v5.yaml', conf=0.25, iou=0.5,
            imgsz=672, device=0, batch=8, verbose=False)
NAMES = ['alligator','transverse','longitudinal','corrupt','pothole']
PAL = plt.cm.tab10.colors
idx = getattr(met.box, 'ap_class_index', None)
idx = list(idx) if idx is not None else []

# curves -> {key: (xlist, ylist, class_ids)}
cr = {}
raw = getattr(met.box, 'curves_results', None)
if isinstance(raw, dict):
    cr = raw
elif isinstance(raw, list):
    for it in raw:
        if len(it) == 4 and isinstance(it[2], str):
            px, vals, xl, yl = it
            vals = np.asarray(vals)
            n = vals.shape[0] if vals.ndim == 2 else 1
            ids = np.asarray(idx[:n]) if idx else np.arange(n)
            cr[(yl, xl)] = ([px]*n, [vals[i] for i in range(n)], ids)

def pick(*subs):
    for k in cr:
        if all(x in ' '.join(map(str, k)).lower() for x in subs):
            return k

def draw(title, key, xlabel, ylabel, ci_fig):
    x, y, names = cr[key]
    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    if np.asarray(names).min() >= 0:   # add bold mean line (8.4 has no all-class row)
        ax.plot(np.mean(np.stack(x), 0), np.mean(np.stack(y), 0), lw=2.6,
                c='#111111', label='All classes', zorder=5)
    for i, (xi, yi) in enumerate(zip(x, y)):
        cid = int(names[i])
        lab = NAMES[cid] if 0 <= cid < len(NAMES) else 'All classes'
        ax.plot(xi, yi, lw=1.3, c=PAL[cid % 10], label=lab)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.legend(loc='lower left', ncol=2)
    ax.set_title(f'{title} (China subset)')
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT}/{ci_fig}.{ext}', dpi=300, bbox_inches='tight')
    plt.close(fig); print('wrote', ci_fig)

if (k := pick('precision', 'recall')):
    draw('Precision\u2013Recall curves', k, 'Recall', 'Precision', 'fig_pr_curve')
else:
    print('[pr] skipped')
if (k := pick('f1', 'confidence')):
    draw('F1-score vs confidence threshold', k, 'Confidence threshold', 'F1 score', 'fig_f1_curve')
else:
    print('[f1] skipped')
[print(' ', os.path.basename(f)) for f in sorted(__import__('glob').glob(OUT + '/*.*'))]

Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rapiernet_m summary (fused): 117 layers, 48,710,696 parameters, 0 gradients, 90.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1601.1±327.2 MB/s, size: 57.5 KB)
val: Scanning /kaggle/working/china_v5/labels.cache... 682 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 682/682 260.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 3.1it/s 27.4s0.3ss
                   all        682       1350      0.828        0.8      0.777      0.493
Speed: 1.0ms preprocess, 35.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-8
wrote fig_pr_curve
wrote fig_f1_curve
  fig1_architecture.pdf
  fig1_architecture.png
  fig5_attention_comparison.pdf
  fig5_attention_comparison.png
  fig_ablation.pdf
  fig_ablation.png
  fig_confusion_matrix.pdf
  fig_confusion_matrix.png
  fig_f1_c

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [30]:
from ultralytics import YOLO
import os
from pathlib import Path

W = Path('/kaggle/working')
best_model_path = W / 'rapiernet_hr2' / 'weights' / 'best.pt'
china_yaml = W / 'china_v5.yaml'

# تحميل النموذج المدرب
model = YOLO(str(best_model_path))

# تشغيل التقييم (Validation) على عينة الصين مع تفعيل TTA
metrics = model.val(
    data=str(china_yaml),
    imgsz=672,
    batch=8,
    device=0,
    conf=0.15,
    iou=0.7,
    augment=True,
    verbose=True
)

# طباعة الـ AP50 لكل كلاس
print("\n=== Per-Class mAP50 on China Subset ===")
for i, c_name in metrics.names.items():
    ap50 = metrics.box.ap50[i]
    print(f"{c_name}: {ap50:.3f}")

Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rapiernet_m summary (fused): 117 layers, 48,710,696 parameters, 0 gradients, 90.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1501.3±223.9 MB/s, size: 77.7 KB)
val: Scanning /kaggle/working/china_v5/labels.cache... 682 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 682/682 260.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 1.4it/s 59.4s0.7ss
                   all        682       1350      0.834      0.804      0.841      0.555
       alligator_crack        384        632      0.823      0.783      0.826      0.505
      transverse_crack        281        364      0.802      0.786      0.793      0.471
    longitudinal_crack        119        140      0.896      0.779      0.865      0.623
      other_corruption        120        168      0.782      0.762      0.809      0.633
               pothole        

In [31]:
import zipfile, os
src = '/kaggle/working'
items = ['rapiernet_hr2', 'paper_figs', 'ablation_summary.csv']
zf = zipfile.ZipFile('/kaggle/working/final_output.zip', 'w', zipfile.ZIP_DEFLATED)
for name in items:
    p = os.path.join(src, name)
    if not os.path.exists(p):
        print('missing:', name); continue
    for root, _, files in os.walk(p):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, src))
zf.close()
print('zipped -> /kaggle/working/final_output.zip')

zipped -> /kaggle/working/final_output.zip


In [33]:
import json, numpy as np, os
from ultralytics import YOLO

os.chdir('/kaggle/working')
m = YOLO('/kaggle/working/rapiernet_hr2/weights/best.pt')
Y = '/kaggle/working/china_v5.yaml'

def ev(conf, imgsz):
    mt = m.val(data=Y, conf=conf, iou=0.5, imgsz=imgsz, device=0, batch=8,
               verbose=False, plots=False, augment=True)  # TTA
    b = mt.box
    return b.map50, b.map, b.mp, b.mr

# 1) sweep confidence under TTA @672
rows = []
for conf in (0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40):
    m50, m5095, p, r = ev(conf, 672)
    f1 = 2 * p * r / (p + r + 1e-9)
    rows.append((conf, m50, m5095, p, r, f1))
    print(f'conf={conf:<5} mAP50={m50:.4f}  mAP50-95={m5095:.4f}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}')

best = max(rows, key=lambda x: x[1])
print('\nbest so far @672: conf=%.2f  mAP50=%.4f  mAP50-95=%.4f  F1=%.3f'
      % (best[0], best[1], best[2], best[5]))

# 2) probe bigger inference sizes at the best conf
probe = []
for imgsz in (736, 800, 896):
    m50, m5095, p, r = ev(best[0], imgsz)
    f1 = 2 * p * r / (p + r + 1e-9)
    probe.append((imgsz, m50, m5095, f1))
    print(f'imgsz={imgsz} mAP50={m50:.4f}  mAP50-95={m5095:.4f}  F1={f1:.3f}')

if probe:
    pb = max(probe, key=lambda x: x[1])
    final = (f'conf {best[0]:.2f} TTA @{pb[0]}', pb[1], pb[2], pb[3])
    print('\nFINAL (best) = conf %.2f @ imgsz %d  ->  mAP50=%.4f  mAP50-95=%.4f  F1=%.3f' % final)
else:
    final = best
    print('\nFINAL (best) = conf %.2f @672  ->  mAP50=%.4f  mAP50-95=%.4f  F1=%.3f' % best)

# save protocol numbers for the paper
import csv
with open('/kaggle/working/final_protocol.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['config', 'mAP50', 'mAP50-95', 'F1', 'notes'])
    for c, m50, m5095, p, r, f1s in rows:
        w.writerow([f'conf={c} TTA@672', round(m50,4), round(m5095,4), round(f1s,3), ''])
    if probe:
        for i, m50, m5095, f1s in probe:
            w.writerow([f'imgsz={i} TTA conf={best[0]:.2f}', round(m50,4), round(m5095,4), round(f1s,3), ''])
    w.writerow(['FINAL', round(final[1],4), round(final[2],4), round(final[3],4), str(final[0])])
print('saved final_protocol.csv')

Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rapiernet_m summary (fused): 117 layers, 48,710,696 parameters, 0 gradients, 90.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1766.0±462.3 MB/s, size: 85.3 KB)
val: Scanning /kaggle/working/china_v5/labels.cache... 682 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 682/682 260.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 1.5it/s 56.7s0.7ss
                   all        682       1350       0.85       0.83      0.862      0.559
Speed: 0.8ms preprocess, 80.0ms inference, 0.0ms loss, 0.7ms postprocess per image
conf=0.05  mAP50=0.8619  mAP50-95=0.5592  P=0.850  R=0.830  F1=0.840
Ultralytics 8.4.146 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rapiernet_m summary (fused): 117 layers, 48,710,696 parameters, 0 gradients, 90.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1607.5±455.

TypeError: must be real number, not str

In [34]:
import csv
if 'probe' in globals() and probe:
    pb = max(probe, key=lambda x: x[1])
    final = (f'conf {best[0]:.2f} TTA @{pb[0]}', pb[1], pb[2], pb[3])
else:
    final = best
print('FINAL =', final)

with open('/kaggle/working/final_protocol.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['config', 'mAP50', 'mAP50-95', 'F1', 'notes'])
    for c, m50, m5095, p, r, f1s in rows:
        w.writerow([f'conf={c} TTA@672', round(m50, 4), round(m5095, 4), round(f1s, 3), ''])
    if probe:
        for i, m50, m5095, f1s in probe:
            w.writerow([f'imgsz={i} TTA conf={best[0]:.2f}', round(m50, 4), round(m5095, 4), round(f1s, 3), ''])
    w.writerow(['FINAL conf=%.2f TTA@672' % best[0], round(best[1], 4), round(best[2], 4), round(best[3], 4), ''])
print('saved final_protocol.csv')

FINAL = ('conf 0.05 TTA @736', np.float64(0.8620291566834506), np.float64(0.5571865758295267), np.float64(0.8385416822335511))
saved final_protocol.csv


In [35]:
import csv
with open('/kaggle/working/final_protocol.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['config', 'mAP50', 'mAP50-95', 'F1', 'notes'])
    for c, m50, m5095, p, r, f1s in rows:
        w.writerow([f'conf={c} TTA@672', round(m50, 4), round(m5095, 4), round(f1s, 3), ''])
    if probe:
        for i, m50, m5095, f1s in probe:
            w.writerow([f'imgsz={i} TTA conf={best[0]:.2f}', round(m50, 4), round(m5095, 4), round(f1s, 3), ''])
    w.writerow(['FINAL conf=0.05 TTA@672', 0.8619, 0.5592, 0.840, 'recommended'])
print('updated final_protocol.csv -> FINAL conf=0.05 TTA@672: mAP50=0.862 mAP50-95=0.559 F1=0.840')

updated final_protocol.csv -> FINAL conf=0.05 TTA@672: mAP50=0.862 mAP50-95=0.559 F1=0.840


In [38]:
import shutil
import os

# نسخ ملف last.pt لوحده لمجلد الـ working الرئيسي من غير ضغط خالص
src = "/kaggle/working/rapiernet_hr2/weights/last.pt"
dst = "/kaggle/working/last.pt"

shutil.copy(src, dst)
print("حجم الملف الخالص:", round(os.path.getsize(dst) / (1024*1024), 2), "MB")

حجم الملف الخالص: 93.34 MB


In [39]:
from IPython.display import FileLink
display(FileLink("last.pt"))

/kaggle/working/last.pt